<a href="https://colab.research.google.com/github/safosaffers/4C_8S_AdMap_VN/blob/main/Welcome_To_Colab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator
import ipywidgets as widgets
from ipywidgets import interact

# -----------------------------------------------------------------------------
# 1. ГЕНЕРАЦИЯ ДАННЫХ И ПОИСК ПИКА (Выполняется 1 раз для скорости)
# -----------------------------------------------------------------------------
grid_size = 120
x_min, x_max = -4.5, 4.5
y_min, y_max = -4.5, 4.5

x = np.linspace(x_min, x_max, grid_size)
y = np.linspace(y_min, y_max, grid_size)
X, Y = np.meshgrid(x, y)

np.random.seed(101)

def fractal_hill_generator(X, Y, octaves=5):
    base_height = 2.4
    base_spread = 6.0
    R2 = X**2 + Y**2
    Z_base = base_height * np.exp(-R2 / base_spread)

    Z_noise = np.zeros_like(X)
    for i in range(octaves):
        freq = 0.7 * (1.9**i)
        amp = 0.35 / (1.8**i)
        angle = np.random.uniform(0, 2*np.pi)
        phase = np.random.uniform(0, 2*np.pi)
        kx = freq * np.cos(angle)
        ky = freq * np.sin(angle)
        Z_noise += amp * np.sin(kx * X + ky * Y + phase)

    return Z_base + Z_noise

Z_terrain = fractal_hill_generator(X, Y)
z_water = 0.2
Z_terrain_water = np.maximum(Z_terrain, z_water)

# 1. Сначала создаем интерполятор (нужен для трассировки лучей intersect_ray)
interp_Z = RegularGridInterpolator((y, x), Z_terrain_water)

# 2. Поиск локального пика в центральной зоне купола (|X| < 1.5, |Y| < 1.5)
center_mask = (np.abs(X) < 1.5) & (np.abs(Y) < 1.5)
Z_center = np.where(center_mask, Z_terrain_water, -np.inf)

max_idx = np.unravel_index(np.argmax(Z_center), Z_center.shape)
x_peak = float(X[max_idx])
y_peak = float(Y[max_idx])
z_peak = float(Z_terrain_water[max_idx])

def intersect_ray(O, V):
    direction = V - O
    t_vals = np.linspace(1.0, 3.5, 300)  # Снизили число точек для быстрой отрисовки
    pts = O + t_vals[:, None] * direction

    valid = (pts[:, 0] >= x.min()) & (pts[:, 0] <= x.max()) & \
            (pts[:, 1] >= y.min()) & (pts[:, 1] <= y.max())
    pts_valid = pts[valid]

    z_terrain = interp_Z((pts_valid[:, 1], pts_valid[:, 0]))
    diff = pts_valid[:, 2] - z_terrain
    idx = np.argmin(np.abs(diff))
    return pts_valid[idx]

def project_boundary(O, pts, samples_per_edge=20):
    boundary = []
    for i in range(len(pts)):
        p_start = pts[i]
        p_end = pts[(i+1)%len(pts)]
        for t in np.linspace(0, 1, samples_per_edge, endpoint=False):
            p_curr = p_start + t * (p_end - p_start)
            boundary.append(intersect_ray(O, p_curr))
    boundary.append(boundary[0])
    return np.array(boundary)

# -----------------------------------------------------------------------------
# 2. ФУНКЦИЯ ОТРИСОВКИ, ПРИВЯЗАННАЯ К ПОЛЗУНКАМ
# -----------------------------------------------------------------------------
def plot_scene(elev=22, azim=-55, h_apex=2.0, w=0.85, h_frame=0.60, show_axis=True):
    fig = plt.figure(figsize=(10, 8), dpi=100)
    ax = fig.add_subplot(111, projection='3d')

    # Геометрия
    O = np.array([x_peak, y_peak, z_peak + h_apex])
    P = np.array([x_peak, y_peak, z_peak])
    z_flat = z_peak

    A = np.array([x_peak - w, y_peak - h_frame, z_flat])
    B = np.array([x_peak + w, y_peak - h_frame, z_flat])
    C = np.array([x_peak + w, y_peak + h_frame, z_flat])
    D = np.array([x_peak - w, y_peak + h_frame, z_flat])

    A_p = intersect_ray(O, A)
    B_p = intersect_ray(O, B)
    C_p = intersect_ray(O, C)
    D_p = intersect_ray(O, D)
    P_p = intersect_ray(O, P)

    red_boundary = project_boundary(O, [A, B, C, D])

    # 1. Поверхность и вода
    ax.plot_surface(X, Y, Z_terrain_water, cmap='gist_earth', alpha=0.6, edgecolor='none', rstride=3, cstride=3)
    X_w, Y_w = np.meshgrid(np.linspace(x_min, x_max, 10), np.linspace(y_min, y_max, 10))
    ax.plot_surface(X_w, Y_w, np.full_like(X_w, z_water), color='deepskyblue', alpha=0.2)
    ax.contour(X, Y, Z_terrain_water, levels=6, zdir='z', offset=z_water, colors='blue', linewidths=0.6, alpha=0.4)

    # 2. Штриховка
    n_hatch = 8
    u_grid = np.linspace(-w, w, n_hatch)
    v_grid = np.linspace(-h_frame, h_frame, n_hatch)

    for u in u_grid:
        p1 = np.array([x_peak + u, y_peak - h_frame, z_flat])
        p2 = np.array([x_peak + u, y_peak + h_frame, z_flat])
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]], color='lightseagreen', lw=0.6, alpha=0.6)
        line_pts = np.array([intersect_ray(O, p1 + t * (p2 - p1)) for t in np.linspace(0, 1, 10)])
        ax.plot(line_pts[:,0], line_pts[:,1], line_pts[:,2], color='red', lw=0.6, alpha=0.6)

    for v in v_grid:
        p1 = np.array([x_peak - w, y_peak + v, z_flat])
        p2 = np.array([x_peak + w, y_peak + v, z_flat])
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]], color='lightseagreen', lw=0.6, alpha=0.6)
        line_pts = np.array([intersect_ray(O, p1 + t * (p2 - p1)) for t in np.linspace(0, 1, 10)])
        ax.plot(line_pts[:,0], line_pts[:,1], line_pts[:,2], color='red', lw=0.6, alpha=0.6)

    # 3. Контуры и грани
    rect_x = [A[0], B[0], C[0], D[0], A[0]]
    rect_y = [A[1], B[1], C[1], D[1], A[1]]
    rect_z = [A[2], B[2], C[2], D[2], A[2]]
    ax.plot(rect_x, rect_y, rect_z, color='mediumturquoise', lw=1.5, label='Кадр ABCD')
    ax.plot(red_boundary[:,0], red_boundary[:,1], red_boundary[:,2], color='red', lw=1.8, label="Проекция A'B'C'D'")

    for V_p in [A_p, B_p, C_p, D_p]:
        ax.plot([O[0], V_p[0]], [O[1], V_p[1]], [O[2], V_p[2]], color='black', lw=0.8)
    ax.plot([O[0], P_p[0]], [O[1], P_p[1]], [O[2], P_p[2]], color='black', lw=0.8, ls='-.')

    # Точки
    ax.scatter(*O, color='black', s=15)
    ax.scatter(*P, color='crimson', s=30, zorder=10)

    # Угол обзора
    ax.view_init(elev=elev, azim=azim)

    if not show_axis:
        ax.set_axis_off()
    else:
        ax.grid(True)
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Z')

    ax.set_title(f"Обзор: elev={elev}°, azim={azim}° (Надир при elev=90)", fontsize=11)
    ax.legend(loc='upper left')
    plt.show()

# -----------------------------------------------------------------------------
# 3. ПОДКЛЮЧЕНИЕ ПОЛЗУНКОВ (IPYWIDGETS)
# -----------------------------------------------------------------------------
interact(
    plot_scene,
    elev=widgets.IntSlider(min=-90, max=90, step=1, value=22, description='Угол elev:'),
    azim=widgets.IntSlider(min=-180, max=180, step=5, value=-55, description='Азимут azim:'),
    h_apex=widgets.FloatSlider(min=0.5, max=5.0, step=0.1, value=2.0, description='Высота O:'),
    w=widgets.FloatSlider(min=0.2, max=2.0, step=0.05, value=0.85, description='Ширина w:'),
    h_frame=widgets.FloatSlider(min=0.2, max=2.0, step=0.05, value=0.60, description='Высота h:'),
    show_axis=widgets.Checkbox(value=True, description='Показать оси')
);

interactive(children=(IntSlider(value=22, description='Угол elev:', max=90, min=-90), IntSlider(value=-55, des…